In [1]:
# ETL Pipeline using Airflow
### Author:[ABIDA HIND]
##
# Description: This script defines and orchestrates an ETL pipeline using Apache Airflow.#
#              It consists of tasks for extracting data frCSV filece# 
#              transforming it as required, and loading it intSqlite databasetion.


In [2]:
import pandas as pd 
import json
from bs4 import BeautifulSoup
import html
import os
from airflow.decorators import dag, task
from airflow.providers.sqlite.hooks.sqlite import SqliteHook
from airflow.providers.sqlite.operators.sqlite import SqliteOperator

In [3]:
# Define paths
SOURCE_CSV = "/mnt/c/Users/Abida Hind/Desktop/data-internship-home-assignment-main/source/jobs.csv"
EXTRACTED_DIR = "/mnt/c/Users/Abida Hind/Desktop/data-internship-home-assignment-main/staging/extracted"
TRANSFORMED_DIR = "/mnt/c/Users/Abida Hind/Desktop/data-internship-home-assignment-main/staging/transformed"
DATABASE_FILE = "/mnt/c/Users/Abida Hind/Desktop/data-internship-home-assignment-main/dags/sqlite_default.db"

In [4]:
@task()
def extract():
    """Extract data from jobs.csv"""
    df = pd.read_csv(SOURCE_CSV)
    df.dropna(axis=0, inplace=True)
    context_data = df['context']

    #save each directory in text file 
    for i, data in enumerate(context_data):
        with open(f"{EXTRACTED_DIR}/extracted_{i}.txt", 'w') as file:
            data[:10]
            file.write(data)  
        file.close() 

extract()

XComArg(<Task(_PythonDecoratedOperator): extract>)

In [5]:
# Clean the job description 
def clean_description(description):
    decoded_text = html.unescape(description)
    soup = BeautifulSoup(decoded_text, 'html.parser')
    cleaned_description = soup.get_text(separator=' ')
    
    return cleaned_description

In [6]:
@task()
def transform():
    """Clean and convert extracted elements to json."""
    os.makedirs(TRANSFORMED_DIR, exist_ok=True)

    for file_name in os.listdir(EXTRACTED_DIR):
        if file_name.endswith(".txt"):
            with open(os.path.join(EXTRACTED_DIR, file_name), 'r') as file:
                data = json.loads(file.read())

            transformed_data = {
                "job": {
                    "title": data.get("title", ""),
                    "industry": data.get("industry", ""),
                    "description": clean_description(data.get("description", "")),
                    "employment_type": data.get("employmentType", ""),
                    "date_posted": data.get("datePosted", ""),
                },
                "company": {
                    "name": data.get("hiringOrganization", {}).get("name", ""),
                    "link": data.get("hiringOrganization", {}).get("sameAs", ""),
                },
                "education": {
                    "required_credential": data.get("educationRequirements", {}).get("credentialCategory", ""),
                },
                "experience": {
                    "months_of_experience": data.get('experienceRequirements', {}).get('monthsOfExperience', '') if isinstance(data.get('experienceRequirements', {}), dict) else '',
                    "seniority_level": None,
                },
                "salary": {
                    "currency": data.get("estimatedSalary", {}).get("currency", ""),
                    "min_value": data.get("estimatedSalary", {}).get("value", {}).get("minValue", ""),
                    "max_value": data.get("estimatedSalary", {}).get("value", {}).get("maxValue", ""),
                    "unit": data.get("estimatedSalary", {}).get("value", {}).get("unitText", ""),
                },
                "location": {
                    "country": data.get("jobLocation", {}).get("address", {}).get("addressCountry", ""),
                    "locality": data.get("jobLocation", {}).get("address", {}).get("addressLocality", ""),
                    "region": data.get("jobLocation", {}).get("address", {}).get("addressRegion", ""),
                    "postal_code": data.get("jobLocation", {}).get("address", {}).get("postalCode", ""),
                    "street_address": data.get("jobLocation", {}).get("address", {}).get("streetAddress", ""),
                    "latitude": data.get("jobLocation", {}).get("latitude", ""),
                    "longitude": data.get("jobLocation", {}).get("longitude", ""),
                },
            }

            transformed_json = json.dumps(transformed_data, indent=2)

            with open(f"{TRANSFORMED_DIR}/transformed_{file_name.split('_')[1].split('.')[0]}.json", 'w') as file:
               
                file.write(transformed_json)

transform()

XComArg(<Task(_PythonDecoratedOperator): transform>)

In [7]:
import sqlite3

@task()
def load():
    """Load data to sqlite database."""
    print("starting loading...")
    db_path = 'sqlite_default.db'  
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # Define  SQLite insert queries 
    job_query = """
        INSERT INTO job (title, industry, description, employment_type, date_posted)
        VALUES (?, ?, ?, ?, ?)
    """
    company_query = """
        INSERT INTO company (name, link)
        VALUES (?, ?)
    """
    education_query = """
        INSERT INTO education (required_credential)
        VALUES (?)
    """
    experience_query = """
        INSERT INTO experience (months_of_experience, seniority_level)
        VALUES (?, ?)
    """
    salary_query = """
        INSERT INTO salary (currency, min_value, max_value, unit)
        VALUES (?, ?, ?, ?)
    """
    location_query = """
        INSERT INTO location (country, locality, region, postal_code, street_address, latitude, longitude)
        VALUES (?, ?, ?, ?, ?, ?, ?)
    """

    for file_name in os.listdir(TRANSFORMED_DIR):
        if file_name.endswith(".json"):
            with open(os.path.join(TRANSFORMED_DIR, file_name), 'r') as file:
                data = json.loads(file.read())

            try:
                # Execute the queries 
                cursor.execute(job_query, (
                    data["job"]["title"],
                    data["job"]["industry"],
                    data["job"]["description"],
                    data["job"]["employment_type"],
                    data["job"]["date_posted"],
                ))

                cursor.execute(company_query, (
                    data["company"]["name"],
                    data["company"]["link"],
                ))

                cursor.execute(education_query, (
                    data["education"]["required_credential"],
                ))

                cursor.execute(experience_query, (
                    data["experience"]["months_of_experience"],
                    data["experience"]["seniority_level"],
                ))

                cursor.execute(salary_query, (
                    data["salary"]["currency"],
                    data["salary"]["min_value"],
                    data["salary"]["max_value"],
                    data["salary"]["unit"],
                ))

                cursor.execute(location_query, (
                    data["location"]["country"],
                    data["location"]["locality"],
                    data["location"]["region"],
                    data["location"]["postal_code"],
                    data["location"]["street_address"],
                    data["location"]["latitude"],
                    data["location"]["longitude"],
                ))

                conn.commit()

            except sqlite3.Error as e:
                print(f"SQLite error: {e}")
                conn.rollback()

    conn.close()
    print("done!")

In [8]:
load()

XComArg(<Task(_PythonDecoratedOperator): load>)

In [17]:
import sqlite3

# SQLite database file path
db_path = 'sqlite_default.db'

# Tables creation query
TABLES_CREATION_QUERY = """CREATE TABLE IF NOT EXISTS job (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    title VARCHAR(225),
    industry VARCHAR(225),
    description TEXT,
    employment_type VARCHAR(125),
    date_posted DATE
);

CREATE TABLE IF NOT EXISTS company (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    job_id INTEGER,
    name VARCHAR(225),
    link TEXT,
    FOREIGN KEY (job_id) REFERENCES job(id)
);

CREATE TABLE IF NOT EXISTS education (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    job_id INTEGER,
    required_credential VARCHAR(225),
    FOREIGN KEY (job_id) REFERENCES job(id)
);

CREATE TABLE IF NOT EXISTS experience (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    job_id INTEGER,
    months_of_experience INTEGER,
    seniority_level VARCHAR(25),
    FOREIGN KEY (job_id) REFERENCES job(id)
);

CREATE TABLE IF NOT EXISTS salary (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    job_id INTEGER,
    currency VARCHAR(3),
    min_value NUMERIC,
    max_value NUMERIC,
    unit VARCHAR(12),
    FOREIGN KEY (job_id) REFERENCES job(id)
);

CREATE TABLE IF NOT EXISTS location (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    job_id INTEGER,
    country VARCHAR(60),
    locality VARCHAR(60),
    region VARCHAR(60),
    postal_code VARCHAR(25),
    street_address VARCHAR(225),
    latitude NUMERIC,
    longitude NUMERIC,
    FOREIGN KEY (job_id) REFERENCES job(id)
);
"""

def create():
    """Create tables in SQLite database."""
    # Connect to SQLite database
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # Execute tables creation query
    cursor.executescript(TABLES_CREATION_QUERY)

    # Commit the changes
    conn.commit()

    # Close the connection
    conn.close()

if __name__ == '__main__':
    # Call the function to create tables
    create_tables()
    print("Tables created successfully!")


Tables created successfully!


In [20]:
import sqlite3

def print_tables(database_path):
    """Print table names from the SQLite database."""
    # Connect to SQLite database
    conn = sqlite3.connect(database_path)
    cursor = conn.cursor()

    # SQL query to select table names from sqlite_master
    query = "SELECT name FROM sqlite_master WHERE type='table';"

    # Execute the query
    cursor.execute(query)

    # Fetch and print table names
    tables = cursor.fetchall()
    for table in tables:
        print(table[0])

    # Close the connection
    conn.close()

if __name__ == '__main__':
    # Path to the SQLite database
    db_path = 'sqlite_default.db'

    # Call the function to print table names
    print_tables(db_path)


job
sqlite_sequence
company
education
experience
salary
location


In [21]:
from datetime import datetime, timedelta

DAG_DEFAULT_ARGS = {
    "depends_on_past": False,
    "retries": 3,
    "retry_delay": timedelta(minutes=15)
}

@dag(
    dag_id="etl_dag",
    description="ETL LinkedIn job posts",
    tags=["etl"],
    schedule="@daily",
    start_date=datetime(2024, 1, 2),
    catchup=False,
    default_args=DAG_DEFAULT_ARGS
)
def etl_dag():
    """ETL pipeline"""

    create_tables = SqliteOperator(
        task_id="create_tables",
        sqlite_conn_id="sqlite_default",
        sql=TABLES_CREATION_QUERY
    )

    create_tables >> extract() >> transform() >> load()

etl_dag()



/tmp/ipykernel_14049/3585993227.py:21 AirflowProviderDeprecationWarning: This class is deprecated.
            Please use `airflow.providers.common.sql.operators.sql.SQLExecuteQueryOperator`.

<DAG: etl_dag>